# 03 · Item-based collaborative filtering

1. Build the user × movie matrix $R$ (943 × 1682) from the training ratings.
2. Cosine similarity between movie columns: $\mathrm{sim}(i,j) = \dfrac{r_{\cdot i}\cdot r_{\cdot j}}{\lVert r_{\cdot i}\rVert\,\lVert r_{\cdot j}\rVert}$ (optionally after subtracting each user's mean — *adjusted cosine*).
3. Keep the $k$ nearest neighbours of each movie.
4. **Ranking:** $s(u,i) = \sum_{j \in I_u} \mathrm{sim}(i,j)\,w_{uj}$, with $w_{uj}=1$ or $w_{uj}=r_{uj}$.
5. **Rating prediction:** $\hat r_{ui} = \bar r_u + \dfrac{\sum_{j \in N_k(i;u)} \mathrm{sim}(i,j)(r_{uj}-\bar r_u)}{\sum_{j} |\mathrm{sim}(i,j)|}$ over the $k$ most similar movies that $u$ rated.

In [ ]:
import sys, json
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.precision", 4); pd.set_option("display.width", 140)
RESULTS = Path("../results"); RESULTS.mkdir(exist_ok=True)

def save_params(name, params):
    path = RESULTS / "best_params.json"
    allp = json.loads(path.read_text()) if path.exists() else {}
    allp[name] = {k: (v.item() if hasattr(v, "item") else v) for k, v in params.items()}
    path.write_text(json.dumps(allp, indent=2))

In [ ]:
from src.data import load_ratings, load_movies, temporal_user_split
ratings, movies = load_ratings(), load_movies()
train, test = temporal_user_split(ratings, test_frac=0.2)        # final evaluation
train_in, valid = temporal_user_split(train, test_frac=0.2)      # hyper-parameter tuning only
title = movies.set_index("item_idx")["title"]
print(f"train={len(train)} test={len(test)} | train_in={len(train_in)} valid={len(valid)}")

In [ ]:
from src.models import ItemKNNRecommender, user_item_matrix
from src.evaluation import grid_search, evaluate_ranking, evaluate_rmse, recommend_top_k
R = user_item_matrix(train)
print("user-item matrix:", R.shape, f"non-zeros={R.nnz}, density={R.nnz / np.prod(R.shape):.3%}")
knn = ItemKNNRecommender(k=50).fit(train)

## Similar movies (cosine on the training matrix)

In [ ]:
def show_similar(model, item_id, n=8):
    idx, sims = model.similar_items(item_id - 1, n)
    return pd.DataFrame({"similar movie": title[idx].values, "cosine": sims})

for item_id in [50, 1, 98]:   # Star Wars (1977), Toy Story (1995), Silence of the Lambs (1991)
    print("\n==", title[item_id - 1]); print(show_similar(knn, item_id).to_string(index=False))

## Hyper-parameter selection on the validation split

In [ ]:
grid_rank = grid_search(ItemKNNRecommender,
                        {"k": [10, 20, 50, 100, 200, 1682], "centered": [False, True],
                         "weighting": ["binary", "rating"]},
                        train_in, valid, objective="ndcg@10", verbose=False)
grid_rank.head(10)

In [ ]:
grid_rmse = grid_search(lambda k, centered: ItemKNNRecommender(k=k, centered=centered),
                        {"k": [10, 20, 50, 100, 200], "centered": [False, True]},
                        train_in, valid, objective="rmse", verbose=False)
grid_rmse

In [ ]:
best_rank = grid_rank.iloc[0][["k", "centered", "weighting"]].to_dict()
best_rmse = grid_rmse.iloc[0][["k", "centered"]].to_dict()
save_params("itemknn_ranking", best_rank); save_params("itemknn_rating", best_rmse)
grid_rank.to_csv(RESULTS / "grid_itemknn_ranking.csv", index=False)
grid_rmse.to_csv(RESULTS / "grid_itemknn_rmse.csv", index=False)
best_rank, best_rmse

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for (c, w), g in grid_rank.groupby(["centered", "weighting"]):
    g = g.sort_values("k"); ax.plot(g.k, g["ndcg@10"], marker="o", label=f"centered={c}, w={w}")
ax.set_xscale("log"); ax.set_xlabel("neighbours k"); ax.set_ylabel("validation NDCG@10"); ax.legend(); ax.grid(alpha=.3);

## Recommendations from a user's history

In [ ]:
model = ItemKNNRecommender(**best_rank).fit(train)
u = 0
hist = train[train.user_idx == u].sort_values("rating", ascending=False)
print("User", u + 1, "top-rated training movies:"); print(title[hist.item_idx.head(5)].to_string())
rec = recommend_top_k(model, train, [u], 10)[0]
liked = set(test[(test.user_idx == u) & (test.rating >= 4)].item_idx)
pd.DataFrame({"recommendation": title[rec].values, "in test & liked": [i in liked for i in rec]}, index=range(1, 11))

## Test-set performance

In [ ]:
rate_model = ItemKNNRecommender(**best_rmse).fit(train)
res = evaluate_ranking(model, train, test, ks=(10,))
res["rmse"] = evaluate_rmse(rate_model, test)
pd.Series(res)